# Seasonal Upset Frequency Data 

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
HERE = Path.cwd()
REPO_ROOT = HERE.parents[3]

DATA_DIR = REPO_ROOT / "data" / "european_soccer_leagues" / "pure_luck_result_based"
OUT_CSV  = HERE / "season_upset_frequency_summary_all_seeds.csv"

LEAGUES = ["bundesliga", "la_liga", "premier_league", "serie_a"]

def load_files(league: str):
    matches = pd.read_csv(DATA_DIR / f"{league}_simulated_matches_all_seeds.csv")
    standings = pd.read_csv(DATA_DIR / f"{league}_simulated_standings_all_seasons.csv")
    return matches, standings

In [7]:
def compute_upset_freq(league: str) -> pd.DataFrame:
    matches, standings = load_files(league)
    seed_cols = [c for c in matches.columns if c.startswith("simulated_home_team_result_seed_")]
    seed_nums = sorted(int(c.split("_")[-1]) for c in seed_cols)

    out_rows = []

    for season, g in matches.groupby("season", sort=True):
        total_matches = len(g)
        rec = {"league": league, "season": int(season), "total_matches": total_matches}

        upset_counts, upset_freqs = [], []

        for n in seed_nums:
            # standings for this seed
            rank_col = f"simulated_rank_{n}"
            ranks = standings[standings["season"] == season][["team", rank_col]]

            # merge ranks into match data
            merged = (
                g.merge(ranks.rename(columns={"team": "home_team", rank_col: "home_rank"}), on="home_team")
                 .merge(ranks.rename(columns={"team": "away_team", rank_col: "away_rank"}), on="away_team")
            )

            sim_col = f"simulated_home_team_result_seed_{n}"
            res = merged[sim_col].to_numpy()
            home_rank = merged["home_rank"].to_numpy()
            away_rank = merged["away_rank"].to_numpy()

            # compute upsets
            upsets = np.zeros(len(res))
            upsets[(res == 1) & (home_rank > away_rank)] = 1.0
            upsets[(res == -1) & (away_rank > home_rank)] = 1.0
            upsets[(res == 0) & (home_rank != away_rank)] = 0.5

            total_upsets = upsets.sum()
            freq = total_upsets / total_matches if total_matches > 0 else np.nan

            rec[f"total_upsets_seed_{n}"] = total_upsets
            rec[f"upset_frequency_seed_{n}"] = freq
            upset_counts.append(total_upsets)
            upset_freqs.append(freq)

        # averages
        rec["total_upsets_avg"] = np.mean(upset_counts)
        rec["upset_frequency_avg"] = np.mean(upset_freqs)

        out_rows.append(rec)

    # column order
    pairs = []
    for n in seed_nums:
        pairs += [f"total_upsets_seed_{n}", f"upset_frequency_seed_{n}"]

    cols = ["league", "season", "total_matches"] + pairs + ["total_upsets_avg", "upset_frequency_avg"]
    return pd.DataFrame(out_rows)[cols]

In [8]:
frames = [compute_upset_freq(lg) for lg in LEAGUES]
combined = pd.concat(frames, ignore_index=True)
combined = combined.sort_values(["league", "season"]).reset_index(drop=True)

combined.to_csv(OUT_CSV, index=False)
print(f"✅ Upset frequency summary written to: {OUT_CSV}")
combined.head()

✅ Upset frequency summary written to: /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/upset_frequency/pure_luck_result_based/season_upset_frequency_summary_all_seeds.csv


,league,season,total_matches,total_upsets_seed_1,upset_frequency_seed_1,total_upsets_seed_2,upset_frequency_seed_2,total_upsets_seed_3,upset_frequency_seed_3,total_upsets_seed_4,...,total_upsets_seed_7,upset_frequency_seed_7,total_upsets_seed_8,upset_frequency_seed_8,total_upsets_seed_9,upset_frequency_seed_9,total_upsets_seed_10,upset_frequency_seed_10,total_upsets_avg,upset_frequency_avg
0,bundesliga,2004,306,110.5,0.361111,137.5,0.449346,138.0,0.450980,126.0,...,118.5,0.387255,124.0,0.405229,131.0,0.428105,131.0,0.428105,126.35,0.412908
1,bundesliga,2005,306,124.0,0.405229,123.0,0.401961,126.5,0.413399,123.5,...,130.5,0.426471,127.5,0.416667,125.5,0.410131,116.0,0.379085,125.05,0.408660
2,bundesliga,2006,306,118.0,0.385621,117.5,0.383987,125.5,0.410131,120.5,...,133.0,0.434641,125.5,0.410131,131.5,0.429739,135.0,0.441176,126.75,0.414216
3,bundesliga,2007,306,136.5,0.446078,134.5,0.439542,136.0,0.444444,128.0,...,132.0,0.431373,133.5,0.436275,130.5,0.426471,134.0,0.437908,132.75,0.433824
4,bundesliga,2008,306,130.5,0.426471,126.5,0.413399,134.5,0.439542,118.5,...,121.0,0.395425,118.0,0.385621,125.0,0.408497,131.0,0.428105,126.45,0.413235


# Overall Upset Frequency Data 

In [9]:
def compute_league_summary(season_df: pd.DataFrame) -> pd.DataFrame:
    """
    Get league level summary totals and frequencies across all seasons.
    """
    seed_nums = sorted({
        int(c.split("_")[-1])
        for c in season_df.columns
        if c.startswith("upset_frequency_seed_")
    })

    records = []
    for lg, g in season_df.groupby("league", sort=True):
        rec = {"league": lg}
        total_matches = g["total_matches"].sum()
        rec["total_matches"] = int(total_matches)

        upset_counts, upset_freqs = [], []

        for n in seed_nums:
            tot_col = f"total_upsets_seed_{n}"
            freq_col = f"upset_frequency_seed_{n}"

            total_upsets = g[tot_col].sum()
            freq = total_upsets / total_matches if total_matches > 0 else np.nan

            rec[f"total_upsets_seed_{n}"] = total_upsets
            rec[f"upset_frequency_seed_{n}"] = freq

            upset_counts.append(total_upsets)
            upset_freqs.append(freq)

        rec["total_upsets_avg"] = np.mean(upset_counts)
        rec["upset_frequency_avg"] = np.mean(upset_freqs)

        records.append(rec)

    pairs = []
    for n in seed_nums:
        pairs += [f"total_upsets_seed_{n}", f"upset_frequency_seed_{n}"]

    cols = ["league", "total_matches"] + pairs + ["total_upsets_avg", "upset_frequency_avg"]
    return pd.DataFrame(records)[cols]

In [10]:
league_summary = compute_league_summary(combined)

OUT_CSV_LEAGUE = HERE / "overall_league_upset_frequency_summary_all_seeds.csv"
league_summary.to_csv(OUT_CSV_LEAGUE, index=False)

print(f"✅ League-level upset summary written to: {OUT_CSV_LEAGUE}")
league_summary

✅ League-level upset summary written to: /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/upset_frequency/pure_luck_result_based/overall_league_upset_frequency_summary_all_seeds.csv


,league,total_matches,total_upsets_seed_1,upset_frequency_seed_1,total_upsets_seed_2,upset_frequency_seed_2,total_upsets_seed_3,upset_frequency_seed_3,total_upsets_seed_4,upset_frequency_seed_4,...,total_upsets_seed_7,upset_frequency_seed_7,total_upsets_seed_8,upset_frequency_seed_8,total_upsets_seed_9,upset_frequency_seed_9,total_upsets_seed_10,upset_frequency_seed_10,total_upsets_avg,upset_frequency_avg
0,bundesliga,6426,2665.0,0.414721,2695.5,0.419468,2724.0,0.423903,2656.0,0.413321,...,2659.0,0.413788,2673.5,0.416044,2685.0,0.417834,2704.5,0.420868,2695.80,0.419514
1,la_liga,8740,3758.5,0.430034,3744.0,0.428375,3702.5,0.423627,3725.5,0.426259,...,3703.5,0.423741,3701.5,0.423513,3730.5,0.426831,3674.5,0.420423,3708.15,0.424273
2,premier_league,8360,3506.0,0.419378,3529.0,0.422129,3538.0,0.423206,3556.0,0.425359,...,3559.0,0.425718,3560.0,0.425837,3552.5,0.424940,3493.0,0.417823,3539.40,0.423373
3,serie_a,8518,3600.0,0.422634,3641.5,0.427506,3636.5,0.426919,3679.5,0.431968,...,3630.0,0.426156,3586.0,0.420991,3659.5,0.429620,3617.5,0.424689,3635.60,0.426814
